*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Learn about Obi-Wan Kenobi 🌌

Are you a Star Wars fan? If not, this exercise will teach you a lot about one of the most important characters of the saga, Obi-Wan Kenobi. We analyse the French Wikipedia article about him: word clouds first, then TF-IDF to find the words that characterize each paragraph, and the cosine similarity to find paragraphs that talk about the same thing.

The text is in French, but you don't need to read French: the word clouds and the numbers will speak for you.

## The data 📜

`data/obi_wan_kenobi_fr.txt` contains the text of the article, **one paragraph per line**.

*Source: Wikipedia, article « Obi-Wan Kenobi », [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

The file was made by downloading the page and keeping the text of its paragraphs (the `<p>` tags of the HTML), with code like this one (you don't need to run it):

```python
import requests
from bs4 import BeautifulSoup

html = requests.get("https://fr.wikipedia.org/wiki/Obi-Wan_Kenobi").text
soup = BeautifulSoup(html, "html.parser")
paragraphs = [p.text.strip() for p in soup.find_all("p") if p.text.strip()]
```

Let's import the libraries and load the paragraphs.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import plotly.express as px
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from spacy.lang.fr.stop_words import STOP_WORDS
from wordcloud import WordCloud

import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"
course_colors = ListedColormap(["#3B47C4", "#E4572E", "#2E7D4F", "#8E7CC3", "#4BA3C3"])

with open("data/obi_wan_kenobi_fr.txt", encoding="utf-8") as file:
    paragraphs = file.read().splitlines()

## Explore the text 🔎

1. How many paragraphs are there? Print the first one. How many words (separated by spaces) do the paragraphs contain: the minimum, the mean and the maximum?

2. Load spaCy's small French model, `fr_core_news_sm`, in a variable `nlp`. Process all the paragraphs with `nlp.pipe` and store the result in a list `docs`. Then display the `text`, `lemma_`, `pos_` and `is_stop` of the first 30 tokens of the first paragraph. What happens to "Obi-Wan", to "l'univers", to the verbs?

💡 Hint: `nlp.pipe(paragraphs)` returns a generator: wrap it in `list(...)` to keep the documents. A slice of a `Doc`, like `docs[0][:30]`, gives its first tokens.

## Word clouds ☁️

3. Make a word cloud of the whole article (all the paragraphs joined in one string). Pass `stopwords=set()` to see the raw text. What do you notice?

💡 Hint: `WordCloud(stopwords=set(), random_state=0).generate(text)` makes the cloud; show it with `plt.imshow(cloud)` and hide the axes with `plt.axis("off")`. Add `background_color="white"` and `colormap=course_colors` if you want.

4. How many French stop words does spaCy know? Print 15 of them (sorted). Then make the word cloud again, with spaCy's French stop words. What changes?

5. Write a function `clean_text(doc, stop_words)` that returns the lemmas of a spaCy `Doc` in lower case, keeping only the tokens made of letters that are not stop words. Apply it to every paragraph and store the result in a list `clean_paragraphs`. Print the first cleaned paragraph, then make the word cloud of the lemmas. What is better than in the previous cloud? Can you spot a mistake of the lemmatizer?

💡 Hint: test both `token.is_stop` and `token.lemma_.lower() not in stop_words`, so that the function also works with your own stop words later.

6. Which are the 20 most frequent lemmas of the article? Show them in a bar chart. Which of them do not teach us anything about Obi-Wan Kenobi?

💡 Hint: split every cleaned paragraph into words, put all the words in a `pd.Series` and use `.value_counts()`.

7. Add "obi", "wan", "kenobi" and a few uninformative words of your choice to the stop words, clean the paragraphs again with this new list, and make a new word cloud. What do you see now?

💡 Hint: `STOP_WORDS | {"obi", "wan"}` is the union of two sets.

## TF-IDF 🍍

For the rest of the exercise, we use `clean_paragraphs` (spaCy's stop words only, **without** our extra ones), to see what TF-IDF does with the name of the hero.

8. First, let's make a common mistake: treat the whole article as **one single document**. Fit a `TfidfVectorizer` on a list that contains only the joined text of all the cleaned paragraphs, and look at the different values of `idf_`. What do you notice? Why? What do the tf-idf weights reduce to?

💡 Hint: `np.unique(vectorizer.idf_)` lists the different values.

9. Now treat **each paragraph as a document**. Fit a `TfidfVectorizer` on `clean_paragraphs`, and print the shape of the matrix. Build a DataFrame with, for each word, its document frequency (the number of paragraphs that contain it) and its idf. Show the 10 words with the smallest idf. What is the idf of "obi", "jedi" and "utapau"?

💡 Hint: `vectorizer.get_feature_names_out()` gives the words in the order of the columns. The document frequency is the number of non-zero values in each column: `(X > 0).sum(axis=0)`, then `np.asarray(...).ravel()` to get a flat array.

10. Compare two rankings of the words: the 15 most frequent words (total number of occurrences, with a `CountVectorizer`) and the 15 words with the highest **mean tf-idf** over the paragraphs. Show them side by side. Which words go down, which words go up?

11. `max_df` removes the words present in too many documents. Fit a `TfidfVectorizer(max_df=0.5)` on the cleaned paragraphs: which words disappear from the vocabulary?

## Similar paragraphs 📐

12. Compute the cosine similarity between all the pairs of paragraphs (a 77 × 77 matrix) and show it as a heatmap. Which two different paragraphs are the most similar? Print them, and find the words that contribute the most to their similarity.

💡 Hint: set the diagonal to 0 with `np.fill_diagonal` (each paragraph is identical to itself), then `np.unravel_index(similarity.argmax(), similarity.shape)` gives the row and the column of the maximum. The TF-IDF rows have a length of 1, so the cosine is the sum of the products of the weights: the largest products are the words that contribute the most.

13. **Bonus: a tiny search engine.** Write a question as a short French text, for example `"combat au sabre laser contre Dark Maul"`. Clean it with the same function, turn it into a TF-IDF vector with the **fitted** vectorizer, and print the 3 paragraphs with the highest cosine similarity to it.

💡 Hint: use `vectorizer.transform([...])`, not `fit_transform`: the query must be described with the vocabulary and the idf learned on the article.